Contexts and recursive pairing.

In [1]:
import requests, readline, sys, json
import numpy
import networkx as nx
import matplotlib as mpl
%matplotlib inline
%config InlineBacked.figure_format = 'svg'
import matplotlib.pyplot as plt
import matplotlib.cm as cmap
import seaborn
from query import *
from contexts import *
from concepts import *

In [2]:
context = makeContext(getQuery(proc_eventQ),'pid','type')
context

{'1046': {'EVENT_CLOSE', 'EVENT_RECVMSG', 'EVENT_SENDMSG'},
 '1197': {'EVENT_READ', 'EVENT_WRITE'},
 '1205': {'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_SENDMSG', 'EVENT_WRITE'},
 '1209': {'EVENT_CLOSE',
  'EVENT_MMAP',
  'EVENT_OPEN',
  'EVENT_READ',
  'EVENT_RECVMSG',
  'EVENT_WRITE'},
 '1212': {'EVENT_CLOSE',
  'EVENT_MMAP',
  'EVENT_OPEN',
  'EVENT_READ',
  'EVENT_WRITE'},
 '1215': {'EVENT_READ', 'EVENT_WRITE'},
 '1216': {'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_SENDMSG', 'EVENT_WRITE'},
 '1343': {'EVENT_WRITE'},
 '1344': {'EVENT_READ'},
 '1470': {'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_WRITE'},
 '1479': {'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_SENDMSG', 'EVENT_WRITE'},
 '1481': {'EVENT_READ', 'EVENT_WRITE'},
 '1573': {'EVENT_CLOSE',
  'EVENT_MMAP',
  'EVENT_OPEN',
  'EVENT_READ',
  'EVENT_RECVMSG',
  'EVENT_SENDMSG'},
 '1590': {'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_SENDMSG', 'EVENT_WRITE'},
 '1603': {'EVENT_CLOSE',
  'EVENT_MMAP',
  'EVENT_OPEN',
  'EVENT_READ',
  'EVENT_RECVMSG',
  'EVENT_WR

In [3]:
concepts = {getConcept(context,{att}) for att in getAtts(context)}
concepts

{frozenset({'EVENT_CONNECT', 'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLONE'}),
 frozenset({'EVENT_FORK'}),
 frozenset({'EVENT_CLOSE'}),
 frozenset({'EVENT_CLOSE',
            'EVENT_CREATE_OBJECT',
            'EVENT_READ',
            'EVENT_WRITE'}),
 frozenset({'EVENT_OPEN'}),
 frozenset({'EVENT_CHANGE_PRINCIPAL',
            'EVENT_CLOSE',
            'EVENT_EXECUTE',
            'EVENT_FORK',
            'EVENT_LOADLIBRARY',
            'EVENT_MMAP',
            'EVENT_MPROTECT',
            'EVENT_OPEN',
            'EVENT_READ'}),
 frozenset({'EVENT_CLOSE',
            'EVENT_CREATE_OBJECT',
            'EVENT_READ',
            'EVENT_RENAME',
            'EVENT_WRITE'}),
 frozenset({'EVENT_EXIT'}),
 frozenset({'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE',
            'EVENT_EXIT',
            'EVENT_MMAP',
            'EVENT_OPEN',
            'EVENT_READ',
            'EVENT_TRUNCATE',
            'EVENT_WRITE'}),
 frozenset({'EVENT_RECVMSG'}),
 froze

In [4]:
extendConcept(context,frozenset())

{frozenset({'EVENT_CONNECT', 'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLONE'}),
 frozenset({'EVENT_FORK'}),
 frozenset({'EVENT_CLOSE'}),
 frozenset({'EVENT_CLOSE',
            'EVENT_CREATE_OBJECT',
            'EVENT_READ',
            'EVENT_WRITE'}),
 frozenset({'EVENT_OPEN'}),
 frozenset({'EVENT_CHANGE_PRINCIPAL',
            'EVENT_CLOSE',
            'EVENT_EXECUTE',
            'EVENT_FORK',
            'EVENT_LOADLIBRARY',
            'EVENT_MMAP',
            'EVENT_MPROTECT',
            'EVENT_OPEN',
            'EVENT_READ'}),
 frozenset({'EVENT_CLOSE',
            'EVENT_CREATE_OBJECT',
            'EVENT_READ',
            'EVENT_RENAME',
            'EVENT_WRITE'}),
 frozenset({'EVENT_EXIT'}),
 frozenset({'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE',
            'EVENT_EXIT',
            'EVENT_MMAP',
            'EVENT_OPEN',
            'EVENT_READ',
            'EVENT_TRUNCATE',
            'EVENT_WRITE'}),
 frozenset({'EVENT_RECVMSG'}),
 froze

In [5]:
testContext = {'a': {'y'}, 'b': {'z'},'c':{'x','y'}}
findAllConcepts(testContext)

{frozenset(),
 frozenset({'y'}),
 frozenset({'z'}),
 frozenset({'x', 'y'}),
 frozenset({'x', 'y', 'z'})}

In [56]:
concepts = findAllConceptsWithMinSupp(context,50)
concepts

{frozenset(),
 frozenset({'EVENT_CLOSE', 'EVENT_EXIT', 'EVENT_FORK'}),
 frozenset({'EVENT_CLOSE', 'EVENT_FORK', 'EVENT_READ'}),
 frozenset({'EVENT_OPEN', 'EVENT_READ', 'EVENT_WRITE'}),
 frozenset({'EVENT_EXIT', 'EVENT_OPEN', 'EVENT_READ'}),
 frozenset({'EVENT_EXIT', 'EVENT_OPEN', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLONE'}),
 frozenset({'EVENT_CLOSE', 'EVENT_MMAP', 'EVENT_READ', 'EVENT_WRITE'}),
 frozenset({'EVENT_EXIT', 'EVENT_MMAP', 'EVENT_READ'}),
 frozenset({'EVENT_FORK'}),
 frozenset({'EVENT_CLOSE'}),
 frozenset({'EVENT_CLOSE', 'EVENT_FORK', 'EVENT_OPEN'}),
 frozenset({'EVENT_OPEN', 'EVENT_READ'}),
 frozenset({'EVENT_MMAP', 'EVENT_OPEN', 'EVENT_READ'}),
 frozenset({'EVENT_MMAP', 'EVENT_WRITE'}),
 frozenset({'EVENT_READ', 'EVENT_RECVMSG'}),
 frozenset({'EVENT_EXIT'}),
 frozenset({'EVENT_CLOSE', 'EVENT_EXIT'}),
 frozenset({'EVENT_CLOSE', 'EVENT_READ'}),
 frozenset({'EVENT_CLOSE', 'EVENT_MMAP', 'EVENT_READ'}),
 frozenset({'EVENT_CLOSE', 'EVENT_EXIT', 'EVENT_MMAP', 'EVENT_READ'}),
 fr

In [38]:
# naive
def successor(concepts,c1,c2):
    return c1 < c2 and {c for c in concepts if c1 < c and c < c2} == set()
def edges(concepts):
    return {(c1,c2) for c1 in concepts for c2 in concepts if successor(concepts,c1,c2)}
edges(concepts)

{(frozenset(), frozenset({'EVENT_READ'})),
 (frozenset(), frozenset({'EVENT_RECVMSG'})),
 (frozenset(), frozenset({'EVENT_CLOSE'})),
 (frozenset(), frozenset({'EVENT_WRITE'})),
 (frozenset(), frozenset({'EVENT_OPEN'})),
 (frozenset(), frozenset({'EVENT_MPROTECT'})),
 (frozenset(), frozenset({'EVENT_MMAP'})),
 (frozenset(), frozenset({'EVENT_FORK'})),
 (frozenset(), frozenset({'EVENT_EXIT'})),
 (frozenset(), frozenset({'EVENT_CLONE'})),
 (frozenset(), frozenset({'EVENT_SENDMSG'})),
 (frozenset({'EVENT_SENDMSG'}), frozenset({'EVENT_RECVMSG', 'EVENT_SENDMSG'})),
 (frozenset({'EVENT_MMAP', 'EVENT_MPROTECT'}),
  frozenset({'EVENT_MMAP', 'EVENT_MPROTECT', 'EVENT_READ'})),
 (frozenset({'EVENT_EXIT', 'EVENT_OPEN'}),
  frozenset({'EVENT_EXIT', 'EVENT_OPEN', 'EVENT_WRITE'})),
 (frozenset({'EVENT_WRITE'}), frozenset({'EVENT_SENDMSG', 'EVENT_WRITE'})),
 (frozenset({'EVENT_READ', 'EVENT_WRITE'}),
  frozenset({'EVENT_READ', 'EVENT_SENDMSG', 'EVENT_WRITE'})),
 (frozenset({'EVENT_CLOSE', 'EVENT_EXIT'}

In [51]:
getSupports(context,concepts)

{frozenset(): 330,
 frozenset({'EVENT_MPROTECT',
            'EVENT_READ',
            'EVENT_RECVMSG',
            'EVENT_SENDMSG',
            'EVENT_WRITE'}): 17,
 frozenset({'EVENT_MMAP', 'EVENT_WRITE'}): 84,
 frozenset({'EVENT_SENDMSG', 'EVENT_WRITE'}): 62,
 frozenset({'EVENT_EXIT', 'EVENT_WRITE'}): 158,
 frozenset({'EVENT_CLOSE', 'EVENT_EXIT', 'EVENT_FORK', 'EVENT_OPEN'}): 44,
 frozenset({'EVENT_CLOSE',
            'EVENT_MMAP',
            'EVENT_OPEN',
            'EVENT_READ',
            'EVENT_RECVMSG'}): 28,
 frozenset({'EVENT_CLOSE', 'EVENT_EXIT', 'EVENT_FORK', 'EVENT_WRITE'}): 46,
 frozenset({'EVENT_MMAP'}): 100,
 frozenset({'EVENT_CLOSE',
            'EVENT_CONNECT',
            'EVENT_FORK',
            'EVENT_MMAP',
            'EVENT_MPROTECT',
            'EVENT_OPEN',
            'EVENT_READ',
            'EVENT_RECVMSG',
            'EVENT_WRITE'}): 8,
 frozenset({'EVENT_CLOSE',
            'EVENT_CONNECT',
            'EVENT_CREATE_OBJECT',
            'EVENT_EXEC

In [42]:
def importance(context,concepts):
    return {concept: len(concept) * getSupport(context,concept) for concept in concepts}
importances = importance(context,concepts)
def snd(x): return x[1]
sorted(importances.items(),key=snd,reverse=True)

[(frozenset({'EVENT_CLONE', 'EVENT_EXIT', 'EVENT_WRITE'}), 339),
 (frozenset({'EVENT_CLOSE',
             'EVENT_MMAP',
             'EVENT_OPEN',
             'EVENT_READ',
             'EVENT_WRITE'}),
  325),
 (frozenset({'EVENT_CLONE', 'EVENT_EXIT'}), 316),
 (frozenset({'EVENT_EXIT', 'EVENT_WRITE'}), 316),
 (frozenset({'EVENT_CLOSE', 'EVENT_MMAP', 'EVENT_OPEN', 'EVENT_READ'}), 308),
 (frozenset({'EVENT_CLONE', 'EVENT_EXIT', 'EVENT_UNIT'}), 306),
 (frozenset({'EVENT_READ', 'EVENT_WRITE'}), 302),
 (frozenset({'EVENT_CLOSE',
             'EVENT_EXIT',
             'EVENT_MMAP',
             'EVENT_OPEN',
             'EVENT_READ',
             'EVENT_WRITE'}),
  300),
 (frozenset({'EVENT_CLOSE',
             'EVENT_EXIT',
             'EVENT_MMAP',
             'EVENT_OPEN',
             'EVENT_READ'}),
  290),
 (frozenset({'EVENT_CLONE', 'EVENT_EXIT', 'EVENT_UNIT', 'EVENT_WRITE'}), 284),
 (frozenset({'EVENT_CLOSE', 'EVENT_OPEN', 'EVENT_READ', 'EVENT_WRITE'}), 272),
 (frozenset({'EVEN

Remove all attribute subsets that are covered by a concept, and count how many attributes are left over.

In [57]:
def abnormality(concepts,attrs):
    matches = {concept for concept in concepts if concept <= attrs}
    for match in matches:
        attrs = attrs - match
    return len(attrs)
{(k,frozenset(context[k])): abnormality(concepts,context[k]) for k in context.keys()}

{('1046', frozenset({'EVENT_CLOSE', 'EVENT_RECVMSG', 'EVENT_SENDMSG'})): 0,
 ('1197', frozenset({'EVENT_READ', 'EVENT_WRITE'})): 0,
 ('1205',
  frozenset({'EVENT_READ',
             'EVENT_RECVMSG',
             'EVENT_SENDMSG',
             'EVENT_WRITE'})): 0,
 ('1209',
  frozenset({'EVENT_CLOSE',
             'EVENT_MMAP',
             'EVENT_OPEN',
             'EVENT_READ',
             'EVENT_RECVMSG',
             'EVENT_WRITE'})): 0,
 ('1212',
  frozenset({'EVENT_CLOSE',
             'EVENT_MMAP',
             'EVENT_OPEN',
             'EVENT_READ',
             'EVENT_WRITE'})): 0,
 ('1215', frozenset({'EVENT_READ', 'EVENT_WRITE'})): 0,
 ('1216',
  frozenset({'EVENT_READ',
             'EVENT_RECVMSG',
             'EVENT_SENDMSG',
             'EVENT_WRITE'})): 0,
 ('1343', frozenset({'EVENT_WRITE'})): 0,
 ('1344', frozenset({'EVENT_READ'})): 0,
 ('1470', frozenset({'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_WRITE'})): 0,
 ('1479',
  frozenset({'EVENT_READ',
             'EVENT_R

In [10]:
{concept for concept in concepts if getSupport(context,concept) > 12}

{frozenset(),
 frozenset({'EVENT_CLOSE', 'EVENT_EXIT', 'EVENT_FORK'}),
 frozenset({'EVENT_MPROTECT',
            'EVENT_READ',
            'EVENT_RECVMSG',
            'EVENT_SENDMSG',
            'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE'}),
 frozenset({'EVENT_CLOSE', 'EVENT_FORK', 'EVENT_READ'}),
 frozenset({'EVENT_MMAP', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE', 'EVENT_MMAP', 'EVENT_READ', 'EVENT_WRITE'}),
 frozenset({'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLONE',
            'EVENT_CLOSE',
            'EVENT_EXIT',
            'EVENT_MMAP',
            'EVENT_READ',
            'EVENT_RECVMSG',
            'EVENT_SENDMSG',
            'EVENT_WRITE'}),
 frozenset({'EVENT_MPROTECT', 'EVENT_WRITE'}),
 frozenset({'EVENT_EXIT',
            'EVENT_MMAP',
            'EVENT_MPROTECT',
            'EVENT_READ',
            'EVENT_WRITE'}),
 frozenset({'EVENT_SENDMSG', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE', 'EVENT_SENDMSG'}),
 frozenset({'EVENT_EXIT', 'EVE

In [11]:
testContext = {'a': {'y'}, 'f':{'x'},'b': {'z'},'c':{'x','y'},'d':{'y'},'e':{'y'}}
findAllConceptsWithMinSupp(testContext,2)

{frozenset(), frozenset({'x'}), frozenset({'y'})}

In [15]:
findAllConceptsWithMinSupp(context,20)

{frozenset(),
 frozenset({'EVENT_CLOSE', 'EVENT_EXIT', 'EVENT_FORK'}),
 frozenset({'EVENT_CLOSE', 'EVENT_FORK', 'EVENT_READ'}),
 frozenset({'EVENT_MMAP', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE', 'EVENT_MMAP', 'EVENT_READ', 'EVENT_WRITE'}),
 frozenset({'EVENT_EXIT',
            'EVENT_MMAP',
            'EVENT_MPROTECT',
            'EVENT_READ',
            'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE'}),
 frozenset({'EVENT_SENDMSG', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE', 'EVENT_SENDMSG'}),
 frozenset({'EVENT_CLOSE',
            'EVENT_MMAP',
            'EVENT_OPEN',
            'EVENT_READ',
            'EVENT_SENDMSG',
            'EVENT_WRITE'}),
 frozenset({'EVENT_EXIT', 'EVENT_FORK'}),
 frozenset({'EVENT_MPROTECT', 'EVENT_WRITE'}),
 frozenset({'EVENT_MPROTECT', 'EVENT_READ', 'EVENT_WRITE'}),
 frozenset({'EVENT_MPROTECT', 'EVENT_READ', 'EVENT_RECVMSG', 'EVENT_WRITE'}),
 frozenset({'EVENT_CLOSE',
            'EVENT_EXECUTE',
            'EVENT_FORK',
            'EVENT_LOADLIB

In [25]:
context = makeContext(getQuery(proc_child_nameQ),'parent','child')


In [28]:
getSupports(context,findAllConceptsWithMinSupp(context,5))

{frozenset(): 52, frozenset({'lesspipe'}): 6, frozenset({'bash'}): 12}

In [35]:
context = makeContext(getQuery(proc_fileQ),'pid','file')
cost(context,{})

53064.866804111109

In [32]:
getSupports(context,findAllConceptsWithMinSupp(context,10))

{frozenset(): 106,
 frozenset({'/etc/ld.so.cache'}): 46,
 frozenset({'/etc/ld.so.cache',
            '/lib/x86_64-linux-gnu/libc.so.6',
            '/lib64/ld-linux-x86-64.so.2'}): 44,
 frozenset({'/dev/pts/6',
            '/etc/ld.so.cache',
            '/lib/x86_64-linux-gnu/libc.so.6',
            '/lib64/ld-linux-x86-64.so.2',
            '/usr/lib/locale/locale-archive'}): 17,
 frozenset({'/etc/ld.so.cache',
            '/lib/x86_64-linux-gnu/libc.so.6',
            '/lib/x86_64-linux-gnu/libdl.so.2',
            '/lib64/ld-linux-x86-64.so.2'}): 16,
 frozenset({'/etc/ld.so.cache',
            '/lib/x86_64-linux-gnu/libc.so.6',
            '/lib/x86_64-linux-gnu/libdl.so.2',
            '/lib64/ld-linux-x86-64.so.2',
            '/usr/lib/locale/locale-archive'}): 14,
 frozenset({'/dev/urandom'}): 10,
 frozenset({'/dev/pts/6',
            '/etc/ld.so.cache',
            '/lib/x86_64-linux-gnu/libc.so.6',
            '/lib64/ld-linux-x86-64.so.2'}): 19,
 frozenset({'/etc/ld.so.cache

In [34]:
context2 = {k:allprefixes(context[k]) for k in context.keys()}
cost(context2,{})

14810.288550975158

In [39]:
getSupports(context2,findAllConceptsWithMinSupp(context2,10))

{frozenset(): 106,
 frozenset({'/etc', '/lib', '/lib/x86_64-linux-gnu', '/lib64'}): 44,
 frozenset({'/home'}): 38,
 frozenset({'/etc', '/lib', '/lib/x86_64-linux-gnu', '/usr'}): 39,
 frozenset({'/dev',
            '/etc',
            '/lib',
            '/lib/x86_64-linux-gnu',
            '/usr',
            '/usr/lib',
            '/usr/lib/x86_64-linux-gnu'}): 11,
 frozenset({'/etc',
            '/home',
            '/home/ta1',
            '/home/ta1/beep_patched_app',
            '/lib',
            '/lib/x86_64-linux-gnu',
            '/lib64'}): 11,
 frozenset({'/etc', '/lib', '/lib/x86_64-linux-gnu', '/lib64', '/usr'}): 38,
 frozenset({'/etc',
            '/lib',
            '/lib/x86_64-linux-gnu',
            '/lib64',
            '/usr',
            '/usr/bin'}): 24,
 frozenset({'/dev',
            '/etc',
            '/lib',
            '/lib/x86_64-linux-gnu',
            '/lib64',
            '/usr',
            '/usr/bin'}): 18,
 frozenset({'/bin',
            '/etc',
  

In [42]:
bagofwords(" a b cde f ",' ')

{'a', 'b', 'cde', 'f'}

In [43]:
allbagofwords({'a b cde f', ' gh i jkl '}, ' ')

{'a', 'b', 'cde', 'f', 'gh', 'i', 'jkl'}

In [47]:
context3 = {k:allbagofwords(context[k],'/') for k in context.keys()}
cost(context3,{})

60486.425652346647

In [49]:
getSupports(context3,findAllConceptsWithMinSupp(context3,25))

{frozenset(): 106,
 frozenset({'bin',
            'dev',
            'etc',
            'ld-linux-x86-64.so.2',
            'ld.so.cache',
            'lib',
            'lib64',
            'libc.so.6',
            'x86_64-linux-gnu'}): 28,
 frozenset({'usr'}): 47,
 frozenset({'lib'}): 50,
 frozenset({'etc', 'ld.so.cache', 'lib', 'x86_64-linux-gnu'}): 46,
 frozenset({'etc',
            'ld-linux-x86-64.so.2',
            'ld.so.cache',
            'lib',
            'lib64',
            'libc.so.6',
            'x86_64-linux-gnu'}): 44,
 frozenset({'etc',
            'ld-linux-x86-64.so.2',
            'ld.so.cache',
            'lib',
            'lib64',
            'libc.so.6',
            'locale',
            'locale-archive',
            'usr',
            'x86_64-linux-gnu'}): 34,
 frozenset({'dev', 'etc'}): 36,
 frozenset({'dev',
            'etc',
            'ld-linux-x86-64.so.2',
            'ld.so.cache',
            'lib',
            'lib64',
            'libc.so.6',
  